# Topological Materials Classification Pipeline: Preprocessing, EDA, SVM, Random Forest & XGBoost


## 1. Load Dataset, Exploratory Data Analysis & Target Encoding


In [ ]:
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.feature_selection import mutual_info_classif
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Mount Google Drive and load topoclass.csv from original Drive path
from google.colab import drive
drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/dataset/topoclass.csv'
df = pd.read_csv(file_path, low_memory=False)

# Encode target column and prepare candidate feature names
exclude_cols = ['ID', 'subset', 'structure', 'MP_ID', 'ICSD_ID', 'TQC_TYPE', 'MAT_TYPE']
feature_candidates = [c for c in df.columns if c not in exclude_cols]
le = LabelEncoder()
df['MAT_TYPE_encoded'] = le.fit_transform(df['MAT_TYPE'].astype(str))
class_names = list(le.classes_)

# Plot target distribution across all topological classes
ax = sns.barplot(x=df['MAT_TYPE'].value_counts().index, y=df['MAT_TYPE'].value_counts().values, palette='mako')
plt.title('Distribution of Material Types (MAT_TYPE)', fontsize=14, weight='bold')
plt.xlabel('Material Class'); plt.ylabel('Count'); plt.tight_layout(); plt.show()
print(f'Dataset: {df.shape[0]:,} rows x {df.shape[1]:,} columns | Classes: {class_names}')


## 2. Pearson Correlation Ranking, Top 300 Selection & Correlation Heatmap


In [ ]:
# Compute Pearson correlation with target and extract top 300 features
numeric_features = df[feature_candidates].apply(pd.to_numeric, errors='coerce')
correlations = numeric_features.corrwith(df['MAT_TYPE_encoded'])
corr_df = pd.DataFrame({'Feature': correlations.index, 'Abs_Corr': correlations.abs().values}).dropna().sort_values(by='Abs_Corr', ascending=False)
top_300 = corr_df.head(300)['Feature'].tolist()

# Plot correlation heatmap for top 15 features against target
top_15 = top_300[:15]
mask = np.triu(np.ones_like(df[['MAT_TYPE_encoded'] + top_15].corr(), dtype=bool))
sns.heatmap(df[['MAT_TYPE_encoded'] + top_15].corr(), mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Correlation Matrix: Target vs. Top 15 Descriptors', fontsize=14, weight='bold')
plt.tight_layout(); plt.show()

## 3. Missing Value Imputation, Stratified Split & Feature Scaling


In [ ]:
# Impute missing values and prepare scaled 80/20 train-test split for top 300 features
X_raw = df[top_300].copy()
for col in top_300:
    if X_raw[col].isnull().sum() > 0:
        X_raw[col] = X_raw[col].fillna(0.0 if ('bond frac' in col.lower() or 'fraction' in col.lower()) else X_raw[col].median())

X_train, X_test, y_train, y_test = train_test_split(X_raw, df['MAT_TYPE_encoded'], test_size=0.20, random_state=42, stratify=df['MAT_TYPE_encoded'])
scaler = StandardScaler()
X_tr = scaler.fit_transform(X_train)
X_te = scaler.transform(X_test)
y_tr, y_te = np.asarray(y_train), np.asarray(y_test)
D = X_tr.shape[1]
sample_weights_tr = compute_sample_weight('balanced', y_tr)
print(f'Train samples: {X_tr.shape[0]:,} | Test samples: {X_te.shape[0]:,} | Total Features: {D}')

## 4. Ant Colony Optimization (ACO) & Binary Firefly Algorithm (BFA) Search


In [ ]:
# Run ACO and Firefly algorithms using mutual information and fast multi-core tree evaluation
rng = np.random.default_rng(42)
eval_idx = rng.choice(len(X_tr), size=min(5000, len(X_tr)), replace=False)
X_eval, y_eval = X_tr[eval_idx], y_tr[eval_idx]

def evaluate_subset(mask):
    if not mask.any(): return 1.0, 0.0
    clf = ExtraTreesClassifier(n_estimators=30, max_depth=12, class_weight='balanced', random_state=42, n_jobs=-1)
    clf.fit(X_eval[:, mask], y_eval)
    macro_f1 = f1_score(y_eval, clf.predict(X_eval[:, mask]), average='macro', zero_division=0)
    return 0.85 * (1.0 - macro_f1) + 0.15 * (mask.sum() / D), macro_f1

# Compute mutual information heuristic and run Ant Colony Optimization (ACO)
mi = mutual_info_classif(X_tr, y_tr, random_state=42)
mi_norm = (mi - mi.min()) / (mi.max() - mi.min() + 1e-12)
eta_in, eta_out = 0.10 + 0.80 * mi_norm, 1.0 - (0.10 + 0.80 * mi_norm)
tau_in, tau_out = np.ones(D), np.ones(D)
best_cost_aco, best_mask_aco = float('inf'), None

for it in range(1, 11):
    ant_res = []
    for _ in range(15):
        prob = (tau_in * (eta_in ** 2.0)) / ((tau_in * (eta_in ** 2.0)) + (tau_out * (eta_out ** 2.0)) + 1e-12)
        m = rng.random(D) < prob
        if not m.any(): m[np.argmax(prob)] = True
        c, f = evaluate_subset(m)
        ant_res.append((m, c))
        if c < best_cost_aco: best_cost_aco, best_mask_aco = c, m.copy()
    tau_in *= 0.75; tau_out *= 0.75
    for m, c in ant_res:
        tau_in[m] += 1.0 / (c + 1e-12); tau_out[~m] += 1.0 / (c + 1e-12)

# Run Binary Firefly Algorithm (BFA) with continuous distance updates and sigmoid thresholding
def sigmoid(x): return 1.0 / (1.0 + np.exp(-np.clip(x, -10, 10)))
positions = rng.uniform(-2.0, 2.0, size=(15, D))
costs = [evaluate_subset(sigmoid(positions[i]) > 0.5)[0] for i in range(15)]
best_idx = np.argmin(costs)
best_cost_fa, best_pos_fa = costs[best_idx], positions[best_idx].copy()

for gen in range(1, 11):
    for i in range(15):
        for j in range(15):
            if costs[j] < costs[i]:
                r = np.linalg.norm(positions[i] - positions[j]) / np.sqrt(D)
                positions[i] += np.exp(-0.5 * (r ** 2)) * (positions[j] - positions[i]) + 0.2 * (rng.random(D) - 0.5)
        m = sigmoid(positions[i]) > 0.5
        if not m.any(): m[np.argmax(sigmoid(positions[i]))] = True
        costs[i] = evaluate_subset(m)[0]
        if costs[i] < best_cost_fa: best_cost_fa, best_pos_fa = costs[i], positions[i].copy()

idx_aco = np.flatnonzero(best_mask_aco)
idx_fa = np.flatnonzero(sigmoid(best_pos_fa) > 0.5)
print(f'ACO Selected: {len(idx_aco)}/{D} features ({(D-len(idx_aco))/D*100:.1f}% reduction) | Cost: {best_cost_aco:.4f}')
print(f'Firefly Selected: {len(idx_fa)}/{D} features ({(D-len(idx_fa))/D*100:.1f}% reduction) | Cost: {best_cost_fa:.4f}')

## 5. Support Vector Machine (SVM): Raw vs. ACO vs. Firefly


In [ ]:
# Train and evaluate Linear SVM models across all 300 features, ACO-selected and Firefly-selected features
svm_raw = LinearSVC(dual=False, C=1.0, class_weight='balanced', max_iter=3000, random_state=42).fit(X_tr, y_tr)
svm_aco = LinearSVC(dual=False, C=1.0, class_weight='balanced', max_iter=3000, random_state=42).fit(X_tr[:, idx_aco], y_tr)
svm_fa = LinearSVC(dual=False, C=1.0, class_weight='balanced', max_iter=3000, random_state=42).fit(X_tr[:, idx_fa], y_tr)

pred_svm_raw, pred_svm_aco, pred_svm_fa = svm_raw.predict(X_te), svm_aco.predict(X_te[:, idx_aco]), svm_fa.predict(X_te[:, idx_fa])
acc_s_raw, f1_s_raw = accuracy_score(y_te, pred_svm_raw), f1_score(y_te, pred_svm_raw, average='macro')
acc_s_aco, f1_s_aco = accuracy_score(y_te, pred_svm_aco), f1_score(y_te, pred_svm_aco, average='macro')
acc_s_fa, f1_s_fa = accuracy_score(y_te, pred_svm_fa), f1_score(y_te, pred_svm_fa, average='macro')

# Display SVM comparison table and side-by-side confusion matrix heatmaps
df_svm = pd.DataFrame({
    'Method': ['Raw (All 300 Features)', f'ACO Optimized ({len(idx_aco)} features)', f'Firefly Optimized ({len(idx_fa)} features)'],
    'Features': [D, len(idx_aco), len(idx_fa)],
    'Accuracy': [f'{acc_s_raw*100:.2f}%', f'{acc_s_aco*100:.2f}%', f'{acc_s_fa*100:.2f}%'],
    'Macro_F1': [f'{f1_s_raw:.4f}', f'{f1_s_aco:.4f}', f'{f1_s_fa:.4f}']
})
display(df_svm)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, (cm, title) in enumerate([
    (confusion_matrix(y_te, pred_svm_raw), f'SVM Raw (300 Feat)\nAcc: {acc_s_raw*100:.1f}% | F1: {f1_s_raw:.3f}'),
    (confusion_matrix(y_te, pred_svm_aco), f'SVM ACO ({len(idx_aco)} Feat)\nAcc: {acc_s_aco*100:.1f}% | F1: {f1_s_aco:.3f}'),
    (confusion_matrix(y_te, pred_svm_fa), f'SVM Firefly ({len(idx_fa)} Feat)\nAcc: {acc_s_fa*100:.1f}% | F1: {f1_s_fa:.3f}')
]):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=axes[i])
    axes[i].set_title(title, fontsize=12, weight='bold'); axes[i].set_xlabel('Predicted'); axes[i].set_ylabel('True')
plt.suptitle('SVM: Confusion Matrix Comparison', fontsize=14, weight='bold', y=1.03); plt.tight_layout(); plt.show()

## 6. Random Forest: Raw vs. ACO vs. Firefly


In [ ]:
# Train and evaluate Random Forest models across all 300 features, ACO-selected and Firefly-selected features
rf_raw = RandomForestClassifier(n_estimators=150, max_depth=20, class_weight='balanced', random_state=42, n_jobs=-1).fit(X_tr, y_tr)
rf_aco = RandomForestClassifier(n_estimators=150, max_depth=20, class_weight='balanced', random_state=42, n_jobs=-1).fit(X_tr[:, idx_aco], y_tr)
rf_fa = RandomForestClassifier(n_estimators=150, max_depth=20, class_weight='balanced', random_state=42, n_jobs=-1).fit(X_tr[:, idx_fa], y_tr)

pred_rf_raw, pred_rf_aco, pred_rf_fa = rf_raw.predict(X_te), rf_aco.predict(X_te[:, idx_aco]), rf_fa.predict(X_te[:, idx_fa])
acc_rf_raw, f1_rf_raw = accuracy_score(y_te, pred_rf_raw), f1_score(y_te, pred_rf_raw, average='macro')
acc_rf_aco, f1_rf_aco = accuracy_score(y_te, pred_rf_aco), f1_score(y_te, pred_rf_aco, average='macro')
acc_rf_fa, f1_rf_fa = accuracy_score(y_te, pred_rf_fa), f1_score(y_te, pred_rf_fa, average='macro')

# Display Random Forest comparison table and side-by-side confusion matrix heatmaps
df_rf = pd.DataFrame({
    'Method': ['Raw (All 300 Features)', f'ACO Optimized ({len(idx_aco)} features)', f'Firefly Optimized ({len(idx_fa)} features)'],
    'Features': [D, len(idx_aco), len(idx_fa)],
    'Accuracy': [f'{acc_rf_raw*100:.2f}%', f'{acc_rf_aco*100:.2f}%', f'{acc_rf_fa*100:.2f}%'],
    'Macro_F1': [f'{f1_rf_raw:.4f}', f'{f1_rf_aco:.4f}', f'{f1_rf_fa:.4f}']
})
display(df_rf)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, (cm, title) in enumerate([
    (confusion_matrix(y_te, pred_rf_raw), f'RF Raw (300 Feat)\nAcc: {acc_rf_raw*100:.1f}% | F1: {f1_rf_raw:.3f}'),
    (confusion_matrix(y_te, pred_rf_aco), f'RF ACO ({len(idx_aco)} Feat)\nAcc: {acc_rf_aco*100:.1f}% | F1: {f1_rf_aco:.3f}'),
    (confusion_matrix(y_te, pred_rf_fa), f'RF Firefly ({len(idx_fa)} Feat)\nAcc: {acc_rf_fa*100:.1f}% | F1: {f1_rf_fa:.3f}')
]):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=class_names, yticklabels=class_names, ax=axes[i])
    axes[i].set_title(title, fontsize=12, weight='bold'); axes[i].set_xlabel('Predicted'); axes[i].set_ylabel('True')
plt.suptitle('Random Forest: Confusion Matrix Comparison', fontsize=14, weight='bold', y=1.03); plt.tight_layout(); plt.show()

## 7. XGBoost: Raw vs. ACO vs. Firefly


In [ ]:
# Train and evaluate XGBoost models across all 300 features, ACO-selected and Firefly-selected features
xgb_raw = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, tree_method='hist', random_state=42, n_jobs=-1, eval_metric='mlogloss').fit(X_tr, y_tr, sample_weight=sample_weights_tr)
xgb_aco = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, tree_method='hist', random_state=42, n_jobs=-1, eval_metric='mlogloss').fit(X_tr[:, idx_aco], y_tr, sample_weight=sample_weights_tr)
xgb_fa = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, tree_method='hist', random_state=42, n_jobs=-1, eval_metric='mlogloss').fit(X_tr[:, idx_fa], y_tr, sample_weight=sample_weights_tr)

pred_xgb_raw, pred_xgb_aco, pred_xgb_fa = xgb_raw.predict(X_te), xgb_aco.predict(X_te[:, idx_aco]), xgb_fa.predict(X_te[:, idx_fa])
acc_x_raw, f1_x_raw = accuracy_score(y_te, pred_xgb_raw), f1_score(y_te, pred_xgb_raw, average='macro')
acc_x_aco, f1_x_aco = accuracy_score(y_te, pred_xgb_aco), f1_score(y_te, pred_xgb_aco, average='macro')
acc_x_fa, f1_x_fa = accuracy_score(y_te, pred_xgb_fa), f1_score(y_te, pred_xgb_fa, average='macro')

# Display XGBoost comparison table and side-by-side confusion matrix heatmaps
df_xgb = pd.DataFrame({
    'Method': ['Raw (All 300 Features)', f'ACO Optimized ({len(idx_aco)} features)', f'Firefly Optimized ({len(idx_fa)} features)'],
    'Features': [D, len(idx_aco), len(idx_fa)],
    'Accuracy': [f'{acc_x_raw*100:.2f}%', f'{acc_x_aco*100:.2f}%', f'{acc_x_fa*100:.2f}%'],
    'Macro_F1': [f'{f1_x_raw:.4f}', f'{f1_x_aco:.4f}', f'{f1_x_fa:.4f}']
})
display(df_xgb)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, (cm, title) in enumerate([
    (confusion_matrix(y_te, pred_xgb_raw), f'XGB Raw (300 Feat)\nAcc: {acc_x_raw*100:.1f}% | F1: {f1_x_raw:.3f}'),
    (confusion_matrix(y_te, pred_xgb_aco), f'XGB ACO ({len(idx_aco)} Feat)\nAcc: {acc_x_aco*100:.1f}% | F1: {f1_x_aco:.3f}'),
    (confusion_matrix(y_te, pred_xgb_fa), f'XGB Firefly ({len(idx_fa)} Feat)\nAcc: {acc_x_fa*100:.1f}% | F1: {f1_x_fa:.3f}')
]):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', xticklabels=class_names, yticklabels=class_names, ax=axes[i])
    axes[i].set_title(title, fontsize=12, weight='bold'); axes[i].set_xlabel('Predicted'); axes[i].set_ylabel('True')
plt.suptitle('XGBoost: Confusion Matrix Comparison', fontsize=14, weight='bold', y=1.03); plt.tight_layout(); plt.show()

## 8. Master Performance Summary (All Models & Selection Methods)


In [ ]:
# Consolidate all model results into a single master benchmark comparison table
master_df = pd.DataFrame([
    {'Model': 'SVM', 'Selection': 'Raw (All 300 Features)', 'Features': D, 'Accuracy': f'{acc_s_raw*100:.2f}%', 'Macro_F1': f'{f1_s_raw:.4f}'},
    {'Model': 'SVM', 'Selection': 'ACO-Optimized', 'Features': len(idx_aco), 'Accuracy': f'{acc_s_aco*100:.2f}%', 'Macro_F1': f'{f1_s_aco:.4f}'},
    {'Model': 'SVM', 'Selection': 'Firefly-Optimized', 'Features': len(idx_fa), 'Accuracy': f'{acc_s_fa*100:.2f}%', 'Macro_F1': f'{f1_s_fa:.4f}'},
    {'Model': 'Random Forest', 'Selection': 'Raw (All 300 Features)', 'Features': D, 'Accuracy': f'{acc_rf_raw*100:.2f}%', 'Macro_F1': f'{f1_rf_raw:.4f}'},
    {'Model': 'Random Forest', 'Selection': 'ACO-Optimized', 'Features': len(idx_aco), 'Accuracy': f'{acc_rf_aco*100:.2f}%', 'Macro_F1': f'{f1_rf_aco:.4f}'},
    {'Model': 'Random Forest', 'Selection': 'Firefly-Optimized', 'Features': len(idx_fa), 'Accuracy': f'{acc_rf_fa*100:.2f}%', 'Macro_F1': f'{f1_rf_fa:.4f}'},
    {'Model': 'XGBoost', 'Selection': 'Raw (All 300 Features)', 'Features': D, 'Accuracy': f'{acc_x_raw*100:.2f}%', 'Macro_F1': f'{f1_x_raw:.4f}'},
    {'Model': 'XGBoost', 'Selection': 'ACO-Optimized', 'Features': len(idx_aco), 'Accuracy': f'{acc_x_aco*100:.2f}%', 'Macro_F1': f'{f1_x_aco:.4f}'},
    {'Model': 'XGBoost', 'Selection': 'Firefly-Optimized', 'Features': len(idx_fa), 'Accuracy': f'{acc_x_fa*100:.2f}%', 'Macro_F1': f'{f1_x_fa:.4f}'}
])
print('=' * 80)
print('MASTER BENCHMARK: SVM vs. RANDOM FOREST vs. XGBOOST')
print('=' * 80)
display(master_df)